# ⚡ Interactive Graph Algorithm & Pathfinding Suite
**DAA Implementation Track** | Visualizer, State Inspector & Complexity Analysis

## Overview
This interactive notebook visualizes fundamental graph traversal and pathfinding algorithms on a top-down hierarchical tree layout:
* **Breadth-First Search (BFS):** Level-by-level navigation tracking Queue state.
* **Depth-First Search (DFS):** Branch-first navigation tracking Stack state.
* **Dijkstra's Shortest Path:** Weighted edge relaxation tracking Priority Queue and distance tables.

---




In [4]:
%pip install networkx matplotlib ipywidgets

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import networkx as nx
import matplotlib.pyplot as plt
from collections import deque
import heapq
import ipywidgets as widgets

# 1. Define Graph with Edge Weights
G = nx.DiGraph()
edges = [
    ('A', 'B', 4), ('A', 'C', 2),
    ('B', 'D', 5), ('B', 'E', 12),
    ('C', 'F', 10), ('C', 'G', 3),
    ('D', 'E', 1), ('F', 'G', 2)
]
for u, v, w in edges:
    G.add_edge(u, v, weight=w)

# 2. Top-Down "Waterfall" Tree Layout
pos = {
    'A': (0, 3),                   # Root
    'B': (-2, 2), 'C': (2, 2),     # Level 1
    'D': (-3, 1), 'E': (-1, 1), 'F': (1, 1), 'G': (3, 1)  # Level 2
}

# 3. Algorithm State Recording Engines
def run_bfs(graph, start='A'):
    visited = set([start])
    queue = deque([start])
    steps = [{'curr': start, 'visited': set(visited), 'ds': list(queue), 'log': f"Started BFS at Root '{start}'"}]
    while queue:
        curr = queue.popleft()
        steps.append({'curr': curr, 'visited': set(visited), 'ds': list(queue), 'log': f"Exploring Node '{curr}'"})
        for nbr in graph.neighbors(curr):
            if nbr not in visited:
                visited.add(nbr)
                queue.append(nbr)
                steps.append({'curr': curr, 'visited': set(visited), 'ds': list(queue), 'log': f"Discovered '{nbr}' -> Added to Queue"})
    return steps

def run_dfs(graph, start='A'):
    visited = set()
    stack = [start]
    steps = [{'curr': start, 'visited': set(visited), 'ds': list(stack), 'log': f"Started DFS at Root '{start}'"}]
    while stack:
        curr = stack.pop()
        if curr not in visited:
            visited.add(curr)
            steps.append({'curr': curr, 'visited': set(visited), 'ds': list(stack), 'log': f"Visited Node '{curr}'"})
            for nbr in list(graph.neighbors(curr))[::-1]:
                if nbr not in visited:
                    stack.append(nbr)
                    steps.append({'curr': curr, 'visited': set(visited), 'ds': list(stack), 'log': f"Pushed '{nbr}' to Stack"})
    return steps

def run_dijkstra(graph, start='A'):
    dist = {node: float('inf') for node in graph.nodes()}
    dist[start] = 0
    pq = [(0, start)]
    visited = set()
    steps = [{'curr': start, 'visited': set(visited), 'dist': dict(dist), 'ds': list(pq), 'log': f"Dijkstra Initialized at '{start}'"}]
    while pq:
        d, curr = heapq.heappop(pq)
        if curr in visited: continue
        visited.add(curr)
        steps.append({'curr': curr, 'visited': set(visited), 'dist': dict(dist), 'ds': list(pq), 'log': f"Extracted Min Node '{curr}' (Dist: {d})"})
        for nbr in graph.neighbors(curr):
            w = graph[curr][nbr]['weight']
            if dist[curr] + w < dist[nbr]:
                dist[nbr] = dist[curr] + w
                heapq.heappush(pq, (dist[nbr], nbr))
                steps.append({'curr': curr, 'visited': set(visited), 'dist': dict(dist), 'ds': list(pq), 'log': f"Relaxed Edge ({curr} -> {nbr}): New Dist = {dist[nbr]}"})
    return steps

# Precompute Steps
algo_data = {
    'BFS (Breadth-First Search)': run_bfs(G),
    'DFS (Depth-First Search)': run_dfs(G),
    "Dijkstra's Shortest Path": run_dijkstra(G)
}

# 4. Interactive Step Renderer
def render_visualization(algorithm, step_idx):
    steps = algo_data[algorithm]
    step_idx = min(step_idx, len(steps) - 1)
    step = steps[step_idx]
    
    fig, ax = plt.subplots(figsize=(8, 5))
    
    # Colors: Active = Red, Visited = Green, Unvisited = Gray
    node_colors = ['#E53935' if n == step['curr'] else '#4CAF50' if n in step['visited'] else '#B0BEC5' for n in G.nodes()]
    
    nx.draw(G, pos, ax=ax, with_labels=True, node_color=node_colors, node_size=900, font_weight='bold', font_color='white', arrowsize=20)
    edge_labels = nx.get_edge_attributes(G, 'weight')
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, ax=ax)
    
    ax.set_title(f"{algorithm} | Step {step_idx + 1} of {len(steps)}", fontsize=12, fontweight='bold')
    plt.show()
    
    print(f"📌 Execution Log: {step['log']}")
    print(f"🔴 Active Node: {step['curr']}")
    print(f"🟢 Visited Set: {sorted(list(step['visited']))}")
    print(f"📦 Data Structure State: {step['ds']}")
    if 'dist' in step:
        print(f"📏 Shortest Distances: {step['dist']}")

# 5. Connect Interactive Controls
algo_select = widgets.Dropdown(options=list(algo_data.keys()), value='BFS (Breadth-First Search)', description='Algorithm:')
step_slider = widgets.IntSlider(min=0, max=len(algo_data['BFS (Breadth-First Search)'])-1, step=1, value=0, description='Step:')

def on_algo_change(change):
    step_slider.max = len(algo_data[change['new']]) - 1
    step_slider.value = 0

algo_select.observe(on_algo_change, names='value')
widgets.interact(render_visualization, algorithm=algo_select, step_idx=step_slider)

interactive(children=(Dropdown(description='Algorithm:', options=('BFS (Breadth-First Search)', 'DFS (Depth-Fi…

<function __main__.render_visualization(algorithm, step_idx)>